# Customer Churn Prediction

In [141]:
import pandas as pd

## 1. Data Understanding & Preparation 

In [142]:
# load dataset
df = pd.read_csv('../data/TelcoCustomerChurn.csv')

### Data type & structure checks

In [143]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [144]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Missing values analysis

In [145]:
# Standard null check
df.isnull().sum()
# => No null values found in the dataset.

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [146]:
# find all string columns and check for empty strings
string_columns = df.select_dtypes(include='object').columns
string_columns_empty_check = df[string_columns].apply(lambda x: (x.str.strip() == '').sum())
string_columns_empty_check
# => TotalCharges has 11 empty strings which needs to be handled.

/tmp/ipykernel_184554/1730311603.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(include='object').columns


customerID           0
gender               0
Partner              0
Dependents           0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
TotalCharges        11
Churn                0
dtype: int64

### Duplicate analysis

In [147]:
# deduplication
print("Duplicate rows:", df.duplicated().sum())

# check for multiple records with same customerID
print("Duplicate customer IDs:", df.duplicated(subset='customerID', keep=False).sum())

cid_unique_count = df['customerID'].nunique()
print("Unique customer IDs:", cid_unique_count)
print("Total rows:", df.shape[0])
# => All customerID are unique, no duplicates found.

Duplicate rows: 0
Duplicate customer IDs: 0
Unique customer IDs: 7043
Total rows: 7043


### Numerical vs Categorial identification

In [148]:
# finalize numerical and categorical columns
numerical_columns = [
    'tenure', 'MonthlyCharges', 'TotalCharges'
]
categorical_columns = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]
target = 'Churn'

### Target-variable analysis

In [149]:
# target analysis
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(2))
# => 73.46% of customers are not churned and 26.54% of customers are churned, data is imbalanced (can be handled in training).

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


### Data cleaning & preprocessing

In [150]:
df['TotalCharges'].isnull().sum()

np.int64(0)

In [151]:
# Fix TotalCharges: convert to numeric, coercing blank strings to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isnull().sum()
# => 11 null values found in TotalCharges after conversion, which were originally empty strings.

np.int64(11)

In [152]:
# These are almost always customers with tenure == 0 (brand new, no charges yet) — verify:
print(df.loc[df['TotalCharges'].isnull(), 'tenure'].value_counts())
# Impute with 0 since it reflects reality (no charges accrued yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

tenure
0    11
Name: count, dtype: int64


In [153]:
# customerID is a unique identifier — no predictive value, drop it
df.drop(columns=['customerID'], inplace=True)

In [154]:
# Defensive: strip stray whitespace from all string columns
obj_cols = df.select_dtypes(include='str').columns
df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())

In [155]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

### Encoding categorical variables

In [156]:
# identify columns
print("Unique values")
for col in df.columns:
    print(f"- {col}")
    unique_values = df[col].nunique()
    print(f"  - Count: {unique_values}")
    if unique_values <= 10:
        print(f"  - Values: {df[col].unique().tolist()}")


Unique values
- gender
  - Count: 2
  - Values: ['Female', 'Male']
- SeniorCitizen
  - Count: 2
  - Values: [0, 1]
- Partner
  - Count: 2
  - Values: ['Yes', 'No']
- Dependents
  - Count: 2
  - Values: ['No', 'Yes']
- tenure
  - Count: 73
- PhoneService
  - Count: 2
  - Values: ['No', 'Yes']
- MultipleLines
  - Count: 3
  - Values: ['No phone service', 'No', 'Yes']
- InternetService
  - Count: 3
  - Values: ['DSL', 'Fiber optic', 'No']
- OnlineSecurity
  - Count: 3
  - Values: ['No', 'Yes', 'No internet service']
- OnlineBackup
  - Count: 3
  - Values: ['Yes', 'No', 'No internet service']
- DeviceProtection
  - Count: 3
  - Values: ['No', 'Yes', 'No internet service']
- TechSupport
  - Count: 3
  - Values: ['No', 'Yes', 'No internet service']
- StreamingTV
  - Count: 3
  - Values: ['No', 'Yes', 'No internet service']
- StreamingMovies
  - Count: 3
  - Values: ['No', 'Yes', 'No internet service']
- Contract
  - Count: 3
  - Values: ['Month-to-month', 'One year', 'Two year']
- PaperlessB

In [157]:
YN_MAP = {'Yes': 1, 'No': 0}
GENDER_MAP = {'Male': 1, 'Female': 0}

In [158]:
# Binary Yes/No columns -> 0/1
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
# SeniorCitizen is already 0/1, so no need to convert it.
for col in binary_cols:
    df[col] = df[col].map(YN_MAP)

df['gender'] = df['gender'].map(GENDER_MAP)
df['Churn'] = df['Churn'].map(YN_MAP)

In [159]:
# multi category columns
multi_cat_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract',
    'PaymentMethod'
]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=False)

In [160]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   int64  
 5   PhoneService                             7043 non-null   int64  
 6   PaperlessBilling                         7043 non-null   int64  
 7   MonthlyCharges                           7043 non-null   float64
 8   TotalCharges                             7043 non-null   float64
 9   Churn                                    7043 non-null   int64  
 10  MultipleLines_No                         7043 non-null   bo

In [161]:
# cast all boolean columns to int (0/1) for consistency
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   int64  
 5   PhoneService                             7043 non-null   int64  
 6   PaperlessBilling                         7043 non-null   int64  
 7   MonthlyCharges                           7043 non-null   float64
 8   TotalCharges                             7043 non-null   float64
 9   Churn                                    7043 non-null   int64  
 10  MultipleLines_No                         7043 non-null   in

### Train-test split

In [162]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(4930, 40) (2113, 40)


In [163]:
# Check the distribution of the target variable in the subsets
print("Training Set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest Set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).mul(100).round(2))

# => Class imbalance is preserved in both training and test sets
# => Good for model evaluation

Training Set:
Churn
0    3622
1    1308
Name: count, dtype: int64
Churn
0    73.47
1    26.53
Name: proportion, dtype: float64

Test Set:
Churn
0    1552
1     561
Name: count, dtype: int64
Churn
0    73.45
1    26.55
Name: proportion, dtype: float64
